In [0]:
catalog = "sandbox"
schema = "bronze"
df_ordenes_bronze = spark.table(f"{catalog}.{schema}.ordenes")

In [0]:
last_process = (spark.sql("""
                          SELECT last_update FROM sandbox.ops.control
                          WHERE table = 'silver.ordenes'
                          """)).collect()
if last_process:
    #df_ordenes_bronze = df_ordenes_bronze.filter(f"fecha_apertura > '{last_process[0].last_update}'")
    df_ordenes_bronze = spark.sql(f"""
                                  select * from sandbox.bronze.ordenes
                                  where fecha_apertura > '{last_process[0].last_update}'
                                  """)

## Duplicados

El método `dropDuplicates` elimina repetidos totalmente identicos, o de acuerdo a un subconjunto de columnas.

Su ejecución devuelve un nuevo DataFrame. Ese resultado se puede guardar en una variable o en la original si se quiere reemplazar.

Como es _Lazy_ no devuelve inmediatamente los datos

In [0]:
display(
    df_ordenes_bronze.dropDuplicates()
    )

In [0]:
display(
    df_ordenes_bronze.dropDuplicates(subset=["orden_id"])
    )

In [0]:
df_ordenes_clean = df_ordenes_bronze.dropDuplicates(subset=["orden_id"])

Las *window functions* permiten realizar operaciones sobre grupos de filas, agrupadas por un campo en particular, por ej país, id, etc.

Para eliminar duplicados, se puede usar una window function como `row_number()` para asignar un número a cada fila dentro de cada grupo de valores repetidos, y luego filtrar solo la primera fila de cada grupo. Así, se eliminan duplicados de manera flexible, incluso cuando se necesita conservar una fila específica según algún criterio (como fecha de creación de modificación)).

In [0]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

window_dedup = Window.partitionBy("orden_id").orderBy("fecha_apertura")

df_ordenes_clean = (
  df_ordenes_bronze.withColumn(
    "row_number",
    row_number().over(window_dedup)
    )
  )

In [0]:
display(df_ordenes_clean)

In [0]:
df_ordenes_clean = df_ordenes_clean.filter("row_number = 1").drop("row_number")

display(df_ordenes_clean)

Fuimos ejecutando operaciones por separado, lo ideal sería encadenarlo así:

```python
window_dedup = Window.partitionBy("orden_id").orderBy("fecha_apertura")
df_ordenes_clean = (
  df_ordenes_bronze.withColumn(
    "row_number",
    row_number().over(window_dedup)
    ).filter("row_number = 1")
    .drop("row_number")
  )
```

## Nulls

El método `dropna` elimina filas que contienen valores nulos en un DataFrame. Se puede usar para limpiar datos, especificando si se deben eliminar filas con cualquier valor nulo (`how="any"`) o solo si todos los valores son nulos (`how="all"`), y también se puede indicar un subconjunto de columnas.

Por último, hay otro parámetro interesante: `thresh`, que permite definir el número mínimo de valores no nulos que una fila debe tener para no ser eliminada.

Al ejecutarse, `dropna` retorna un nuevo DataFrame. Es importante guardar el resultado en una variable para conservar los datos limpios. Recordar también que es una operación _lazy_.


In [0]:
df_ordenes_clean = df_ordenes_clean.dropna(subset=["orden_id"])
display(df_ordenes_clean)

El método `fillna` permite imputar valores nulos en un DataFrame, reemplazándolos por una constante o por un valor calculado (como la media, mediana, valor mas frecuente, etc.).

In [0]:
null_mapping = {
    "prioridad": "desconocida",
    # "resolucion": "pendiente",
    "fecha_cierre": "1900-01-01 00:00:00"
}

df_ordenes_clean = df_ordenes_clean.fillna(null_mapping)


display(df_ordenes_clean)

```python
df.withColumn(
  "column_name",
  when(resolucion is null AND/OR cierre is null, "pendiente")
  .when(cierre is not null AND resolucion is null, "error/desconocido, analizar")
  .otherwise # si no hay, se pondrá null

)
```

In [0]:
# la función when de pyspark se usa como un CASE
# para retornar un valor según la condición

# la función col de pyspark se usa en condiciones para referenciar 
# a la columna del df por su nombre
from pyspark.sql.functions import when, col

df_ordenes_clean = df_ordenes_clean.withColumn(
    "resolucion",
    when(
        # Si la resolucion Y la fecha de cierre son nulas, la resolucion es pendiente
        col("resolucion").isNull() & col("fecha_cierre").isNull(),
        "pendiente"
    ).when(
        # si tiene fecha de cierre pero no resolucion, entonces orden sin resolucion
        col("resolucion").isNull() & col("fecha_cierre").isNotNull(),
        "sin_resolucion"
    # caso contrario, ya tiene resolucion, se mantiene el valor original
    ).otherwise(col("resolucion"))
)

In [0]:
# la funcion avg de pyspark calcula el promedio de una columna (previamente debe estar agrupada)
# sino calcula el promedio de la columna entera
# coalesce de pyspark permite retornar el primer valor no nulo de una lista de columnas
# si todas son nulas, retorna null

from pyspark.sql.functions import avg, coalesce

# Vamos a reemplazar los valores nulos, o cero, 

# Definimos una ventana para agrupar las ordenes por equipo
window_avg = Window.partitionBy("equipo_id")

#
(
    # Creamos una columna auxiliar y temporal
    df_ordenes_clean.withColumn(
    "costo_estimado_mean",
    avg("costo_estimado").over(window_avg)
    ).withColumn(
        "costo_estimado",
        when(col("costo_estimado") == 0, col("costo_estimado_mean"))
        .otherwise(col("costo_estimado"))
    ).drop("costo_estimado_mean")
)

## Corección de tipos de datos

PySpark tiene un módulo, `pyspark.sql.types`, donde contiene los tipos de datos soportados, como `StringType`, `IntegerType`, `FloatType`, etc.

```python
df.withColumn(
  "column_name",
  col(column_name).cast("asd")
)
```

In [0]:
cast_rules = {
    "costo_estimado": "double",
    "fecha_apertura": "timestamp",
    "fecha_cierre": "timestamp"
}

for col_name, col_type in cast_rules.items():
    print(f"Casteando columna {col_name} a {col_type}")
    df_ordenes_clean = df_ordenes_clean.withColumn(
        col_name,
        col(col_name).cast(col_type)
    )

## Limpieza de strings

In [0]:
from pyspark.sql.functions import trim, upper, regexp_replace

# display(df_ordenes_clean.withColumn(
#   "prioridad",
#   # Vamos a encadenar varias transformaciones en una misma expression
#   regexp_replace(
#     upper(trim(col("prioridad"))),
#     r"[-\s]", # expresion o patron para detectar guiones o espacios en blanco dentro del texto
#     "", # y reemplazarlos por nada
#   )
# ).limit(10))

df_ordenes_clean = df_ordenes_clean.withColumn(
  "prioridad",
  # Vamos a encadenar varias transformaciones en una misma expression
  regexp_replace(
    upper(trim(col("prioridad"))),
    r"[-\s]", # expresion o patron para detectar guiones o espacios en blanco dentro del texto
    "", # y reemplazarlos por nada
  )
)

In [0]:
# Unificar IDs a mayuscula y asegurar formato
id_cols = ["equipo_id", "orden_id"]
pattern_id = [r"^EQ-\d{4}$", r"^OT-\d{5}$"]

for col_id, pattern in zip(id_cols, pattern_id):
  df_ordenes_clean = df_ordenes_clean.withColumn(
      col_id,
      upper(col(col_id))
  )

  df_ordenes_clean = df_ordenes_clean.filter(
      col(col_id).rlike(pattern)
  )

## Outliers

In [0]:
percentiles = df_ordenes_clean.approxQuantile("costo_estimado", [0.25, 0.75], 0.0)
q1, q3 = percentiles
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
df_ordenes_clean = df_ordenes_clean.filter(
    f"costo_estimado BETWEEN {lower_bound} AND {upper_bound}"
)

In [0]:
df_ordenes_clean = df_ordenes_clean.selectExpr(
  "orden_id", "equipo_id", "costo_estimado", "fecha_apertura", "fecha_cierre", "prioridad", "resolucion", "tecnico_asignado", "tipo_trabajo")
df_ordenes_clean.createOrReplaceTempView("ordenes_clean")

## Columnas derivadas


## Almacenamiento

In [0]:
last_update = df_ordenes_clean.selectExpr("MAX(fecha_apertura)").collect()[0][0]


In [0]:
from delta.tables import DeltaTable

delta_table_control = DeltaTable.forName(spark, "sandbox.ops.control")
delta_table_control.update(
  condition = "table = 'silver.ordenes'",
  set = {
    "last_update": f"'{last_update}'"
    }
  )

In [0]:
%sql
MERGE INTO sandbox.silver.ordenes AS target
USING ordenes_clean AS source
ON target.orden_id = source.orden_id
WHEN NOT MATCHED THEN INSERT (orden_id, equipo_id, costo_estimado, fecha_apertura, fecha_cierre, prioridad, resolucion, tecnico_asignado, tipo_trabajo) VALUES (source.orden_id, source.equipo_id, source.costo_estimado, source.fecha_apertura, source.fecha_cierre, source.prioridad, source.resolucion, source.tecnico_asignado, source.tipo_trabajo)

In [0]:
%sql
select * from sandbox.silver.ordenes